In [ ]:
# 1. Imports and setup
from romansh_lemmatizer import Lemmatizer, Lemma
from collections import Counter
import pandas as pd
from pathlib import Path
from datasets import load_dataset
import re

mediomatix = load_dataset("ZurichNLP/mediomatix", split="train[:5000]")
idioms = ["rm-sursilv", "rm-sutsilv", "rm-surmiran", "rm-puter", "rm-vallader"]


In [22]:
# 2. Inspect and assign text
"""
inspection_data = mediomatix[0]
for k, v in inspection_data.items():
    print(k, v)
print()
"""
corpus = {k:[] for k in idioms}
for row in mediomatix:
    for idiom in idioms:
        if row.get(idiom):
            sent = row[idiom]
            sent = re.sub(r"</?strong>", " ", sent) # Remove <strong> tags
            sent = re.sub(r'["“”«»‚‘’„][^"“”«»‚‘’„]+["“”«»‚‘’„]', " ", sent) # Remove quoted substrings (straight, angled, curly quotes)
            corpus[idiom].append(sent)

for k, v in corpus.items():
    print(k, len(v))

rm-sursilv 99
rm-sutsilv 98
rm-surmiran 0
rm-puter 81
rm-vallader 81


In [26]:
# 3. Initialize lemmatizer (specific idioms)
lemmatizers = {idiom: Lemmatizer(idiom=idiom) for idiom in idioms}

In [47]:
from collections import Counter

# Build per-idiom lemma + translation counters
corpus_lemma_counter = {k: Counter() for k in idioms}

for idiom, sentences in corpus.items():
    for sent in sentences:
        doc = lemmatizers[idiom](sent)
        for t in doc.tokens:
            lemma_objs = list(t.lemmas.keys())
            if lemma_objs:
                lemma_obj = lemma_objs[0]
                lemma_text = lemma_obj.text
                lemma_trans = lemma_obj.translation_de
            else:
                lemma_text = t.text
                lemma_trans = None

            corpus_lemma_counter[idiom][(lemma_text, lemma_trans)] += 1

In [51]:
for idiom, counter in corpus_lemma_counter.items():
    print(f"\n===== {idiom} =====")

    # Filter out short lemmas (≤ 1 char)
    filtered = {
        (lemma, trans): freq
        for (lemma, trans), freq in counter.items()
        if lemma and len(lemma) > 1
    }

    if not filtered:
        print("(no lemmas found)")
        continue

    # Find max widths for alignment
    max_lemma_len = max(len(lemma) for lemma, _ in filtered.keys())
    max_trans_len = max(len(trans or "") for _, trans in filtered.keys())

    # Header
    print(f"{'Lemma'.ljust(max_lemma_len)}   {'German'.ljust(max_trans_len)}   Freq")
    print("-" * (max_lemma_len + max_trans_len + 10))

    # Print top lemmas
    for (lemma, trans), freq in Counter(filtered).most_common(20):
        trans_str = trans if trans and trans != "null" else ""
        print(f"{lemma.ljust(max_lemma_len)}   {trans_str.ljust(max_trans_len)}   {freq}")



===== rm-sursilv =====
Lemma                             German                                                    Freq
------------------------------------------------------------------------------------------------
el                                                                                          39
ella                              #el                                                       25
haver                             haben                                                     24
esser                             sein                                                      21
in                                eins                                                      14
ch'                                                                                         14
cura                              Kur                                                       14
bugen                             gern                                                      14
il                    